# Objective: run a conservative student preview

Validate a canonical fixture, then calculate a small fixture preview. A full ACTIVSg2000 preview requires an instructor-approved scenario artifact; this checkout does not contain that runtime artifact, so the notebook says so instead of inventing a score. `ACTIVSG2000_ARTIFACT_DIR` is informational unless a preview integration explicitly consumes it; this notebook remains fixture-only.


In [ ]:
from pathlib import Path
import csv, json, os, sys

TUTORIAL_ROOT = Path(os.environ.get('TUTORIAL_ROOT', Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('TUTORIAL_OUTPUT_ROOT', TUTORIAL_ROOT / 'outputs')).resolve(); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIXTURE_ROOT = OUTPUT_ROOT / 'fixtures'; FIXTURE_ROOT.mkdir(parents=True, exist_ok=True)
csv_path = FIXTURE_ROOT / 'preview_submission.csv'
columns = ['team_id', 'site_id', 'bus_id', 'connection_capacity_mw', 'generator_capacity_mw', 'storage_power_mw', 'storage_energy_mwh']
with csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle); writer.writerow(columns); writer.writerow(['fixture-team', 'site-north', '1001', 40, 20, 5, 20])


In [ ]:
print('Environment report')
print({'python': sys.version.split()[0], 'preview': 'conservative and non-authoritative', 'fixture_first': True, 'artifact_dir_policy': 'informational unless explicitly consumed'})


In [ ]:
from ercot_game.csv_validator import validate_csv_submission
report = validate_csv_submission(csv_path, bus_ids={'1001', '1002'}, horizon=4368)
assert report.valid, [issue.display() for issue in report.issues]
runtime_artifact = os.environ.get('ACTIVSG2000_ARTIFACT_DIR')
runtime_available = bool(runtime_artifact and Path(runtime_artifact).is_dir())
assert len(columns) == 7 and report.site_count == 1
if runtime_available:
    print('Instructor-approved runtime artifact supplied; full preview integration is intentionally opt-in.')
else:
    print('Runtime artifact unavailable; using deterministic fixture preview only.')
fixture_preview_score = 40.0 - 20.0 * 0.1 - 5.0 * 0.02
assert fixture_preview_score > 0


In [ ]:
OUTPUT = OUTPUT_ROOT / 'student_preview.json'
payload = {'mode': 'fixture_student_preview', 'exact': False, 'runtime_artifact_available': runtime_available, 'horizon_hours': 4368, 'checked_period_count': 0, 'fixture_preview_score': fixture_preview_score, 'warning': 'Preview is conservative and non-authoritative; use the instructor evaluator for official scores.', 'csv_valid': report.valid}
OUTPUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('WARNING: preview is conservative and non-authoritative.')
print('OUTPUT:', OUTPUT)
